In [ ]:
# @title 🚀 1 · Setup & Mount Drive { display-mode: "form" }
from google.colab import drive
import os, time, shutil, requests, base64
!pip install -q runpod

print("📂 Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)

# @title ⚙️ 2 · Configuration { display-mode: "form" }
RUNPOD_API_KEY = "YOUR_API_KEY_HERE" # @param {type:"string"}
ENDPOINT_ID = "YOUR_ENDPOINT_ID_HERE" # @param {type:"string"}

# Paths (Match your original workflow)
DRIVE_BASE_PATH = "/content/drive/Shareddrives/Figuro/multi-angle-shots" # @param {type:"string"}
INPUT_FOLDER = f"{DRIVE_BASE_PATH}/input"
OUTPUT_FOLDER = f"{DRIVE_BASE_PATH}/output"
PROCESSED_FOLDER = f"{DRIVE_BASE_PATH}/processed"

# Choose your Tool
MODE = "Qwen Multi-Angle" # @param ["Qwen Multi-Angle", "Fast Upscale", "High-Quality Upscale"]

# Mapping Logic
configs = {
    "Qwen Multi-Angle": {"file": "qwen.json", "node": 25},
    "Fast Upscale": {"file": "fast_upscale.json", "node": 15},
    "High-Quality Upscale": {"file": "high_upscale.json", "node": 16}
}

import runpod
runpod.api_key = RUNPOD_API_KEY
endpoint = runpod.Endpoint(ENDPOINT_ID)

def image_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode('utf-8')

# @title 🏃‍♂️ 3 · Start Batch Processing { display-mode: "form" }
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(PROCESSED_FOLDER, exist_ok=True)

print(f"👀 Watching {INPUT_FOLDER}...")

while True:
    files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
    
    if not files:
        print("☕ No images found. Waiting 10s...", end="\r")
        time.sleep(10)
        continue

    for filename in files:
        input_path = os.path.join(INPUT_FOLDER, filename)
        print(f"\n🚀 Sending {filename} to RunPod ({MODE})...")
        
        # Convert local Drive image to Base64 so RunPod can see it
        base64_image = image_to_base64(input_path)

        try:
            # Call RunPod
            job = endpoint.run({
                "workflow": configs[MODE]["file"],
                "images": [
                    {
                        "node_id": configs[MODE]["node"],
                        "image": f"data:image/png;base64,{base64_image}"
                    }
                ]
            })

            print(f"⏳ Processing {filename} (GPU is working)...")
            result = job.output() # This waits for the 45-60s generation

            # Save results back to Drive
            if "images" in result:
                for i, img_data in enumerate(result["images"]):
                    # If RunPod returns a URL, download it
                    img_res = requests.get(img_data)
                    output_name = f"{os.path.splitext(filename)[0]}_out_{i}.png"
                    with open(os.path.join(OUTPUT_FOLDER, output_name), "wb") as f:
                        f.write(img_res.content)
                    print(f"✅ Saved to Drive: {output_name}")

                # Move original to processed
                shutil.move(input_path, os.path.join(PROCESSED_FOLDER, filename))
                print(f"📦 Moved {filename} to processed folder.")

        except Exception as e:
            print(f"❌ Error with {filename}: {e}")

    time.sleep(5)